# ML-KEM 768 — HLS IP Execution Time Benchmark
## Software-Level Performance Measurement (Mức Phần mềm)

Measures **only** the HLS IP execution time: `AP_START` → `AP_DONE`.

### ❌ Excluded from measurement:
- Overlay loading time
- `pynq.allocate()` buffer allocation
- Data generation / copy to buffer / `flush()` / `invalidate()`
- Register programming (`_write_pointer`)
- Result readback

### ✅ Included (unavoidable AXI overhead):
- AXI-Lite register write latency (AP_START) ~100ns
- AXI-Lite register read latency (polling AP_DONE) ~100ns × N polls
- Linux OS scheduling jitter

### Method:
- `time.perf_counter()` — highest resolution timer in Python
- **Tight busy-wait** (no `sleep()`) for minimum polling overhead

## 1. Imports & Constants

In [ ]:
import numpy as np
import time
import os

from pynq import Overlay, allocate

# Kyber-768 sizes
PK_SIZE   = 1184
SK_SIZE   = 2400
CT_SIZE   = 1088
SS_SIZE   = 32
SEED_SIZE = 32

# IP dùng AXI‑Lite, mỗi thanh ghi rộng 32‑bit (4 byte)
# AXI-Lite control
REG_CTRL = 0x00
AP_START = 0x01
AP_DONE  = 0x02
AP_IDLE  = 0x04

# KeyGen register offsets
KEYGEN_REG_SEED_D = 0x10 #16
KEYGEN_REG_SEED_Z = 0x1C #28
KEYGEN_REG_PK_OUT = 0x28 #40
KEYGEN_REG_SK_OUT = 0x34 #52

# Encaps register offsets
ENCAPS_REG_PK_IN  = 0x10 #(16 bytes Offset Dec)
# 0x14	20	(reserved)	—
# 0x18	24	(reserved)	—
ENCAPS_REG_RAND_M = 0x1C #(28 bytes Offset Dec)
# 0x20	32	(reserved)	—
# 0x24	36	(reserved)	—
ENCAPS_REG_CT_OUT = 0x28 #(40 bytes Offset Dec)
# 0x2C	44	(reserved)	—
# 0x30	48	(reserved)	—
ENCAPS_REG_SS_OUT = 0x34 #(52 bytes Offset Dec)

# * Có khoảng trống (reserved) → để:
    # căn chỉnh bus
    # mở rộng IP sau này
    # tránh lỗi truy cập không hợp lệ

# Decaps register offsets
DECAPS_REG_SK_IN  = 0x10
DECAPS_REG_CT_IN  = 0x1C
DECAPS_REG_SS_OUT = 0x28

# Benchmark config
N_RUNS = 10

# Bitstream directory
script_dir = os.path.dirname(os.path.abspath("__file__"))
BIT_DIR = os.path.join(script_dir, "..", "bitstream")

print(f"Bitstream dir: {os.path.abspath(BIT_DIR)}")
print(f"N_RUNS = {N_RUNS}")

## 2. Helper Functions

In [ ]:
def _write_pointer(ip, offset_lo, addr):
    """Write a 64-bit physical address into two 32-bit AXI-Lite registers."""
    ip.write(offset_lo, addr & 0xFFFFFFFF)
    ip.write(offset_lo + 4, (addr >> 32) & 0xFFFFFFFF)


def _bytes_to_u64_array(data: bytes) -> np.ndarray:
    """Convert raw bytes (multiple of 8) to uint64 array."""
    assert len(data) % 8 == 0
    return np.frombuffer(data, dtype=np.uint64).copy()


def _perf_start_and_wait(ip, timeout_sec=30.0):
    """Start IP and tight-poll AP_DONE.
    
    Returns execution time in seconds.
    Uses time.perf_counter() — highest resolution.
    NO sleep() — pure busy-wait for minimum overhead.
    Measures ONLY: AP_START write → AP_DONE detected.
    """
    t_start = time.perf_counter()
    ip.write(REG_CTRL, AP_START)
    while True:
        ctrl = ip.read(REG_CTRL)
        if ctrl & AP_DONE:
            t_end = time.perf_counter()
            return t_end - t_start
        if time.perf_counter() - t_start > timeout_sec:
            raise TimeoutError(
                f"HLS IP timeout after {timeout_sec}s. CTRL=0x{ctrl:08X}"
            )


def print_stats(name, times):
    """Print min/avg/max/std statistics for a list of times."""
    avg = np.mean(times) * 1000
    mn  = np.min(times) * 1000
    mx  = np.max(times) * 1000
    std = np.std(times) * 1000
    print(f"  → {name} Avg: {avg:.4f} ms | Min: {mn:.4f} ms | "
          f"Max: {mx:.4f} ms | Std: {std:.4f} ms")
    return avg, mn, mx, std


print("Helpers defined: _write_pointer, _bytes_to_u64_array, _perf_start_and_wait, print_stats")

## 3. KeyGen Benchmark

In [ ]:
print("=" * 60)
print("  KeyGen Benchmark")
print("=" * 60)

# Load overlay ONCE (NOT timed)
keygen_bit = os.path.join(BIT_DIR, "Keygen", "ml_kem_keygen_0.bit")
ol = Overlay(keygen_bit)
ip = ol.ml_kem_keygen_0

# Allocate buffers ONCE (NOT timed)
seed_d_buf = allocate(shape=(4,), dtype=np.uint64)
seed_z_buf = allocate(shape=(4,), dtype=np.uint64)
pk_buf = allocate(shape=(PK_SIZE,), dtype=np.uint8)
sk_buf = allocate(shape=(SK_SIZE,), dtype=np.uint8)

keygen_times = []
for run in range(N_RUNS):
    # Prepare data (NOT timed)
    seed_d_buf[:] = _bytes_to_u64_array(os.urandom(32))
    seed_z_buf[:] = _bytes_to_u64_array(os.urandom(32))
    seed_d_buf.flush()
    seed_z_buf.flush()
    _write_pointer(ip, KEYGEN_REG_SEED_D, seed_d_buf.physical_address)
    _write_pointer(ip, KEYGEN_REG_SEED_Z, seed_z_buf.physical_address)
    _write_pointer(ip, KEYGEN_REG_PK_OUT, pk_buf.physical_address)
    _write_pointer(ip, KEYGEN_REG_SK_OUT, sk_buf.physical_address)

    # ═══ TIMED: AP_START → AP_DONE only ═══
    dt = _perf_start_and_wait(ip)
    keygen_times.append(dt)

    # Readback (NOT timed)
    pk_buf.invalidate()
    sk_buf.invalidate()
    print(f"  Run {run+1:2d}/{N_RUNS}: {dt*1000:.4f} ms")

# Cleanup
seed_d_buf.freebuffer()
seed_z_buf.freebuffer()
pk_buf.freebuffer()
sk_buf.freebuffer()
ol.free()

kg_stats = print_stats("KeyGen", keygen_times)

## 4. Encaps Benchmark

In [ ]:
print("=" * 60)
print("  Encaps Benchmark")
print("=" * 60)

encaps_bit = os.path.join(BIT_DIR, "Encaps", "ml_kem_encaps_0.bit")
ol = Overlay(encaps_bit)
ip = ol.ml_kem_encaps_0

pk_buf   = allocate(shape=(PK_SIZE,), dtype=np.uint8)
rand_buf = allocate(shape=(SEED_SIZE,), dtype=np.uint8)
ct_buf   = allocate(shape=(CT_SIZE,), dtype=np.uint8)
ss_buf   = allocate(shape=(SS_SIZE,), dtype=np.uint8)

encaps_times = []
for run in range(N_RUNS):
    pk_buf[:] = np.frombuffer(os.urandom(PK_SIZE), dtype=np.uint8)
    rand_buf[:] = np.frombuffer(os.urandom(SEED_SIZE), dtype=np.uint8)
    pk_buf.flush()
    rand_buf.flush()
    _write_pointer(ip, ENCAPS_REG_PK_IN,  pk_buf.physical_address)
    _write_pointer(ip, ENCAPS_REG_RAND_M, rand_buf.physical_address)
    _write_pointer(ip, ENCAPS_REG_CT_OUT, ct_buf.physical_address)
    _write_pointer(ip, ENCAPS_REG_SS_OUT, ss_buf.physical_address)

    dt = _perf_start_and_wait(ip)
    encaps_times.append(dt)

    ct_buf.invalidate()
    ss_buf.invalidate()
    print(f"  Run {run+1:2d}/{N_RUNS}: {dt*1000:.4f} ms")

pk_buf.freebuffer()
rand_buf.freebuffer()
ct_buf.freebuffer()
ss_buf.freebuffer()
ol.free()

en_stats = print_stats("Encaps", encaps_times)

## 5. Decaps Benchmark

In [ ]:
print("=" * 60)
print("  Decaps Benchmark")
print("=" * 60)

decaps_bit = os.path.join(BIT_DIR, "Decaps", "ml_kem_decaps_0.bit")
ol = Overlay(decaps_bit)
ip = ol.ml_kem_decaps_0

sk_buf = allocate(shape=(SK_SIZE,), dtype=np.uint8)
ct_buf = allocate(shape=(CT_SIZE,), dtype=np.uint8)
ss_buf = allocate(shape=(SS_SIZE,), dtype=np.uint8)

decaps_times = []
for run in range(N_RUNS):
    sk_buf[:] = np.frombuffer(os.urandom(SK_SIZE), dtype=np.uint8)
    ct_buf[:] = np.frombuffer(os.urandom(CT_SIZE), dtype=np.uint8)
    sk_buf.flush()
    ct_buf.flush()
    _write_pointer(ip, DECAPS_REG_SK_IN,  sk_buf.physical_address)
    _write_pointer(ip, DECAPS_REG_CT_IN,  ct_buf.physical_address)
    _write_pointer(ip, DECAPS_REG_SS_OUT, ss_buf.physical_address)

    dt = _perf_start_and_wait(ip)
    decaps_times.append(dt)

    ss_buf.invalidate()
    print(f"  Run {run+1:2d}/{N_RUNS}: {dt*1000:.4f} ms")

sk_buf.freebuffer()
ct_buf.freebuffer()
ss_buf.freebuffer()
ol.free()

dc_stats = print_stats("Decaps", decaps_times)

## 6. Summary Table

In [ ]:
print("=" * 64)
print("  SUMMARY — IP Execution Time (Software-level measurement)")
print("=" * 64)
print(f"  {'Kernel':<10} {'Avg (ms)':>10} {'Min (ms)':>10} {'Max (ms)':>10} {'Std (ms)':>10}")
print(f"  {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")

for name, stats in [("KeyGen", kg_stats), ("Encaps", en_stats), ("Decaps", dc_stats)]:
    avg, mn, mx, std = stats
    print(f"  {name:<10} {avg:>10.4f} {mn:>10.4f} {mx:>10.4f} {std:>10.4f}")

print(f"  {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
total_avg = kg_stats[0] + en_stats[0] + dc_stats[0]
print(f"  {'TOTAL':<10} {total_avg:>10.4f}")
print()
print(f"  N_RUNS = {N_RUNS} | Clock = pl_clk0 @ 100 MHz")
print(f"  Method: time.perf_counter() [AP_START → AP_DONE]")
print(f"  Note: Includes AXI-Lite R/W latency + OS scheduling jitter")